# Phase 4 — LLM Fine-tuning (PEFT · LoRA · QLoRA)

**SupportAI · Notebook 05** — run on **Google Colab (T4 GPU)**.

### Where we are
Phases 0–3 established that classification is essentially *solved* on this dataset: TF‑IDF + Logistic Regression hits **0.995 macro‑F1** and fine‑tuned DistilBERT reaches **~0.999**. So the interesting question for Phase 4 is no longer "can we classify better?" — it's **"what does it cost to bring a generative LLM to this problem, and is it worth it?"**

### What this notebook does
1. **Response generation** (the genuinely new capability): fine‑tune an LLM to *write* a support reply from a customer query, using **QLoRA** (4‑bit).
2. **Two model sizes**: **Phi‑2 (2.7B)** to validate the pipeline cheaply, then **Mistral‑7B‑Instruct** for the headline result.
3. **LLM classification**: zero‑shot and few‑shot classification with the instruction‑tuned model, compared head‑to‑head with the BERT baseline.
4. **LoRA vs QLoRA**: the efficiency tradeoff (VRAM, speed, quality).

### Deliverables
- ROUGE scores + sample generations (base vs fine‑tuned)
- LLM‑vs‑BERT classification comparison
- A LoRA/QLoRA tradeoff table written to `experiments/phase4_results.csv`
- Saved LoRA adapters + model cards under `models/`

> **Runtime**: target a T4 (16 GB). QLoRA keeps a 7B model under ~6–7 GB for weights; subsample sizes below are tuned to finish in a Colab session. Bump them up if you have an A100.

## 1. Setup

Install the PEFT/QLoRA stack and confirm a GPU is attached. All tunable knobs (model names, subsample sizes, LoRA rank, learning rate) live in the constants cell so the rest of the notebook reads cleanly.

In [ ]:
# =============================================================
# 1a. Install the PEFT / QLoRA stack (Colab)
# =============================================================
# Pinned to a known-compatible set. trl/peft/transformers move fast and
# break each other often — if you bump one, expect to bump the others.
# After this cell finishes you may need: Runtime > Restart session.
%pip install -q \
    "transformers==4.44.2" \
    "peft==0.13.2" \
    "trl==0.11.4" \
    "bitsandbytes==0.43.3" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "evaluate==0.4.3" \
    "rouge_score==0.1.2" \
    "sentencepiece"

print("✅ Install complete. If imports below fail, restart the runtime and re-run from here.")

In [ ]:
# =============================================================
# 1b. Imports, GPU check, and all knobs in one place
# =============================================================
import torch, random, numpy as np

# --- Reproducibility ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Models ---
PHI2_MODEL    = "microsoft/phi-2"                       # 2.7B, open, pipeline validation
MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"    # 7B, the headline run
# NOTE: Mistral may require accepting terms on the HF Hub.

# --- Output dirs (created on demand; flat for Colab) ---
MODELS_DIR      = _OUT_MODELS = "models"
EXPERIMENTS_DIR = "experiments"

# --- Data subsampling (tuned for a T4 Colab session; raise on an A100) ---
N_TRAIN_GEN = 2000     # generation training examples
N_EVAL      = 200      # generation eval (ROUGE) pool
N_CLS_EVAL  = 330      # classification eval tickets (~30/category x 11)

# --- Sequence / generation lengths ---
MAX_SEQ_LEN    = 512
MAX_NEW_TOKENS = 128

# --- SFT hyperparameters ---
EPOCHS        = 1
BATCH_SIZE    = 2
GRAD_ACCUM    = 8      # effective batch = 16
LEARNING_RATE = 2e-4   # typical for LoRA/QLoRA (higher than full-FT)

# --- LoRA ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# --- GPU check ---
assert torch.cuda.is_available(), (
    "No CUDA GPU detected. On Colab: Runtime > Change runtime type > GPU (T4). "
    "QLoRA/bitsandbytes require CUDA.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | {gpu.total_memory/1e9:.1f} GB | torch {torch.__version__}")
print(f"Models: {PHI2_MODEL} , {MISTRAL_MODEL}")

## 2. Data & instruction formatting

Load the cleaned dataset and the Phase-1 `label_encoder`, then rebuild the **same 70/15/15 stratified split** used in notebooks 02–04 (seed 42) so results are comparable across phases. Then shape two task views: **generation** (query → reference reply) and a balanced **classification** eval set.

> **Colab setup**: upload `customer_support_clean.csv` and `label_encoder.pkl` to the session (or mount Drive). The loader below also finds them under `../data` / `../models` when run locally.

In [ ]:
# =============================================================
# 2a. Load cleaned data + label encoder, rebuild the Phase-3 splits
# =============================================================
import os, pandas as pd, joblib
from sklearn.model_selection import train_test_split

def _find(*candidates):
    """Colab keeps files flat; locally they live under ../data, ../models."""
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]   # fall through to the first (will raise a clear error on read)

DATA_CSV = _find("customer_support_clean.csv", "../data/customer_support_clean.csv", "data/customer_support_clean.csv")
LE_PATH  = _find("label_encoder.pkl", "../models/label_encoder.pkl", "models/label_encoder.pkl")
print(f"data  : {DATA_CSV}")
print(f"encoder: {LE_PATH}")

df = pd.read_csv(DATA_CSV)
label_encoder = joblib.load(LE_PATH)
CATEGORIES = list(label_encoder.classes_)
df["label"] = label_encoder.transform(df["category"])

# Identical split recipe to notebooks 02–04 (70/15/15, stratified, seed 42)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"\nRows: {len(df)} | categories: {len(CATEGORIES)}")
print(f"Split: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"Categories: {CATEGORIES}")
df[["instruction", "response", "category"]].head(3)

In [ ]:
# =============================================================
# 2b. Instruction-format datasets (generation + classification)
# =============================================================
# Formatting is model-specific (Mistral has a chat template, Phi-2
# does not), so we keep raw (instruction, response) frames here and
# let run_sft() apply each model's template at train time via
# build_text(). We only subsample once, deterministically.

# The instruction that frames the generation task:
GEN_INSTRUCTION = (
    "You are a helpful customer-support agent. Write a concise, polite "
    "reply to the following customer message.\n\nCustomer: {query}"
)

# Generation: train on (query -> reference reply)
gen_train_df = (train_df.sample(n=min(N_TRAIN_GEN, len(train_df)), random_state=42)
                [["instruction", "response"]].reset_index(drop=True))
gen_eval_df = (test_df.sample(n=min(N_EVAL, len(test_df)), random_state=42)
               [["instruction", "response"]].reset_index(drop=True))

# Classification eval: balanced-ish sample across categories from TEST
per_cat = max(1, N_CLS_EVAL // len(CATEGORIES))
cls_eval_df = (test_df.groupby("category", group_keys=False)
               .apply(lambda g: g.sample(n=min(len(g), per_cat), random_state=42))
               [["instruction", "category"]].reset_index(drop=True))

print(f"Generation : {len(gen_train_df)} train / {len(gen_eval_df)} eval")
print(f"Classification eval: {len(cls_eval_df)} tickets (~{per_cat}/category)\n")
print("Example training target (fallback format shown; chat models reformat at train time):")
print("-" * 60)
print(f"Instruct: {GEN_INSTRUCTION.format(query=gen_train_df.iloc[0]['instruction'])}")
print(f"Output: {gen_train_df.iloc[0]['response'][:200]}...")

## 3. PEFT helpers (QLoRA / LoRA / SFT)

One set of functions reused by every model below, so Phi-2 and Mistral run **identical** code — only the model name and the `quantized` flag change. This is what makes the LoRA-vs-QLoRA and Phi-2-vs-Mistral comparisons fair.

- `load_causal_lm(name, quantized)` — 4-bit NF4 (QLoRA) or fp16 (plain LoRA)
- `run_sft(...)` — trl `SFTTrainer`, returns trainer + wall-time + peak VRAM
- `generate / evaluate_rouge` — greedy decoding + ROUGE-L vs reference
- `finalize_run(...)` — save adapter + register for the model card

In [ ]:
# =============================================================
# 3. PEFT helpers — model loading, LoRA, SFT, generation, ROUGE
# =============================================================
# These functions are shared by every model/section below so the
# Phi-2 and Mistral runs use *identical* code (only the model name
# and quantized flag change). trl/peft APIs evolve quickly — pinned
# versions are in the install cell
import os, json, time, torch
import evaluate as hf_evaluate
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

ADAPTER_REGISTRY = {}      # name -> {base_model, task, metrics, dir}
TOKENIZER_CACHE = {}       # model_name -> tokenizer
_rouge = hf_evaluate.load("rouge")


def build_text(tokenizer, user_content, assistant_content=None):
    """Prompt formatter. Uses the model's chat template when it has one
    (Mistral-Instruct), else a Phi-2-style Instruct/Output fallback.
    assistant_content=None -> inference prompt (ends ready for generation)."""
    if getattr(tokenizer, "chat_template", None):
        msgs = [{"role": "user", "content": user_content}]
        if assistant_content is None:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        msgs.append({"role": "assistant", "content": assistant_content})
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    if assistant_content is None:
        return f"Instruct: {user_content}\nOutput:"
    return f"Instruct: {user_content}\nOutput: {assistant_content}{tokenizer.eos_token}"


def load_causal_lm(name, quantized=True):
    """Load a causal LM. quantized=True -> 4-bit NF4 (QLoRA); False -> fp16 (plain LoRA)."""
    tok = TOKENIZER_CACHE.get(name)
    if tok is None:
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        tok.padding_side = "right"
        TOKENIZER_CACHE[name] = tok
    common = dict(trust_remote_code=True, device_map="auto")
    if quantized:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, **common)
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, **common)
    model.config.use_cache = False     # required for gradient checkpointing during training
    return model, tok


def get_lora_config():
    return LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules="all-linear")


def make_sft_dataset(tokenizer, train_df):
    """Format (instruction -> response) pairs with THIS model's template.
    Done per-model because Mistral uses a chat template and Phi-2 does not."""
    texts = [build_text(tokenizer, GEN_INSTRUCTION.format(query=q), a)
             for q, a in zip(train_df["instruction"].tolist(), train_df["response"].tolist())]
    return Dataset.from_dict({"text": texts})


def run_sft(model, tokenizer, train_df, run_name):
    """Supervised fine-tune on train_df's (instruction, response) pairs.
    Returns (trainer, seconds, peak_VRAM_GB)."""
    train_ds = make_sft_dataset(tokenizer, train_df)
    torch.cuda.reset_peak_memory_stats()
    args = SFTConfig(
        output_dir=f"outputs/{run_name}",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=25,
        save_strategy="no",
        fp16=True,
        optim="paged_adamw_8bit",
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none",
    )
    try:
        trainer = SFTTrainer(model=model, train_dataset=train_ds, args=args,
                             peft_config=get_lora_config(), processing_class=tokenizer)
    except TypeError:   # older trl uses `tokenizer=` instead of `processing_class=`
        trainer = SFTTrainer(model=model, train_dataset=train_ds, args=args,
                             peft_config=get_lora_config(), tokenizer=tokenizer)
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9
    return trainer, dt, peak


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


def evaluate_rouge(model, tokenizer, df, n=80):
    """Generate replies for n queries and score ROUGE vs the reference response."""
    sub = df.head(n)
    preds, refs, samples = [], [], []
    for q, ref in zip(sub["instruction"].tolist(), sub["response"].tolist()):
        prompt = build_text(tokenizer, GEN_INSTRUCTION.format(query=q))
        pred = generate(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS)
        preds.append(pred); refs.append(ref)
        if len(samples) < 3:
            samples.append((q, ref, pred))
    res = _rouge.compute(predictions=preds, references=refs)
    return {"rouge1": res["rouge1"], "rouge2": res["rouge2"],
            "rougeL": res["rougeL"], "samples": samples}


def finalize_run(trainer, name, base_model, task, metrics):
    """Save the LoRA adapter and register the run for the model-card cell."""
    out_dir = os.path.join(MODELS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    trainer.model.save_pretrained(out_dir)             # adapter only (~MBs)
    try:
        TOKENIZER_CACHE[base_model].save_pretrained(out_dir)
    except Exception:
        pass
    ADAPTER_REGISTRY[name] = {"base_model": base_model, "task": task,
                              "metrics": metrics, "dir": out_dir}
    print(f"  saved adapter -> {out_dir}")

print("✅ Helpers defined: load_causal_lm, run_sft, generate, evaluate_rouge, finalize_run")

## 4. Phi-2 — response generation (QLoRA)

Validate the whole fine-tuning loop on the cheap model first. If Phi-2 trains, generates, and scores cleanly, the identical code scales to the 7B in section 5. We capture Phi-2's QLoRA VRAM/time here so section 8 can compare it against fp16 LoRA.

In [ ]:
# =============================================================
# 4. Phi-2 (2.7B) — response generation (QLoRA), pipeline validation
# =============================================================
# Phi-2 is small and fast: we use it to prove the SFT pipeline end to
# end before paying for the 7B run. We record its QLoRA VRAM/time
# (phi2_qlora_stats) for the LoRA-vs-QLoRA comparison in section 8.
import gc, torch

torch.cuda.reset_peak_memory_stats()
p_model, p_tok = load_causal_lm(PHI2_MODEL, quantized=True)

print("Scoring BASE Phi-2 (before fine-tuning)...")
p_base = evaluate_rouge(p_model, p_tok, gen_eval_df, n=min(80, N_EVAL))

print("Fine-tuning Phi-2 with QLoRA...")
p_trainer, p_time, p_vram = run_sft(p_model, p_tok, gen_train_df, run_name="phi2_qlora_gen")

print("Scoring FINE-TUNED Phi-2...")
p_ft = evaluate_rouge(p_trainer.model, p_tok, gen_eval_df, n=min(80, N_EVAL))

phi2_gen_scores = {
    "rougeL_base": p_base["rougeL"],
    "rougeL_ft":   p_ft["rougeL"],
    "samples":     p_ft["samples"],
}
phi2_qlora_stats = {"peak_vram_gb": p_vram, "train_time_s": p_time}   # reused in section 8
finalize_run(p_trainer, "phi2_qlora_gen", PHI2_MODEL, "response generation",
             {"method": "QLoRA 4-bit", "rougeL_base": round(p_base["rougeL"], 4),
              "rougeL_ft": round(p_ft["rougeL"], 4),
              "peak_vram_gb": round(p_vram, 2), "train_time_s": round(p_time, 1)})

print(f"\nPhi-2  ROUGE-L: base={p_base['rougeL']:.4f} -> fine-tuned={p_ft['rougeL']:.4f}")
print(f"Peak VRAM (QLoRA): {p_vram:.2f} GB | train time: {p_time:.1f}s")

del p_model, p_trainer
gc.collect(); torch.cuda.empty_cache()

## 5. Mistral-7B-Instruct — response generation (QLoRA)

The headline run. Same SFT pipeline as Phi-2, on a real 7B model, made to fit a T4 by **4-bit QLoRA**. We score base vs fine-tuned with ROUGE-L, save the adapter, and free VRAM before the classification section reloads the model.

In [ ]:
# =============================================================
# 5. Mistral-7B-Instruct — response generation (QLoRA)
# =============================================================
# Same pipeline, the real 7B model. QLoRA (4-bit) is what makes this
# fit on a T4. We save the adapter inline and free the model before
# the classification section reloads it (T4 VRAM is tight).
import gc, torch

torch.cuda.reset_peak_memory_stats()
m_model, m_tok = load_causal_lm(MISTRAL_MODEL, quantized=True)

print("Scoring BASE Mistral (before fine-tuning)...")
m_base = evaluate_rouge(m_model, m_tok, gen_eval_df, n=min(80, N_EVAL))

print("Fine-tuning Mistral-7B with QLoRA...")
m_trainer, m_time, m_vram = run_sft(m_model, m_tok, gen_train_df, run_name="mistral_qlora_gen")

print("Scoring FINE-TUNED Mistral...")
m_ft = evaluate_rouge(m_trainer.model, m_tok, gen_eval_df, n=min(80, N_EVAL))

mistral_gen_scores = {
    "rougeL_base": m_base["rougeL"],
    "rougeL_ft":   m_ft["rougeL"],
    "samples":     m_ft["samples"],          # list of (query, reference, prediction)
}
finalize_run(m_trainer, "mistral_qlora_gen", MISTRAL_MODEL, "response generation",
             {"method": "QLoRA 4-bit", "rougeL_base": round(m_base["rougeL"], 4),
              "rougeL_ft": round(m_ft["rougeL"], 4),
              "peak_vram_gb": round(m_vram, 2), "train_time_s": round(m_time, 1)})

print(f"\nMistral-7B  ROUGE-L: base={m_base['rougeL']:.4f} -> fine-tuned={m_ft['rougeL']:.4f}")
print(f"Peak VRAM during QLoRA training: {m_vram:.2f} GB | train time: {m_time:.1f}s")

# Free VRAM for the next section
del m_model, m_trainer
gc.collect(); torch.cuda.empty_cache()

## 6. Generation quality — ROUGE

ROUGE-L measures overlap with the reference reply (per `METRICS.md`). We report **base vs fine-tuned** for both models — the *delta* is the real story: how much fine-tuning moves a model toward the dataset's response style. ROUGE is a rough proxy; the qualitative examples and the human-eval rubric (Phase 5) matter more for a generation task.

In [ ]:
# =============================================================
# 6. Consolidated generation quality (ROUGE) + qualitative look
# =============================================================
import pandas as pd

gen_summary = pd.DataFrame([
    {"model": "Phi-2 (2.7B)",        "ROUGE-L base": phi2_gen_scores["rougeL_base"],
     "ROUGE-L fine-tuned": phi2_gen_scores["rougeL_ft"],
     "delta": phi2_gen_scores["rougeL_ft"] - phi2_gen_scores["rougeL_base"]},
    {"model": "Mistral-7B-Instruct", "ROUGE-L base": mistral_gen_scores["rougeL_base"],
     "ROUGE-L fine-tuned": mistral_gen_scores["rougeL_ft"],
     "delta": mistral_gen_scores["rougeL_ft"] - mistral_gen_scores["rougeL_base"]},
]).round(4)
print("Response-generation quality (ROUGE-L, higher = closer to reference reply):")
print(gen_summary.to_string(index=False))

# Qualitative side-by-side on a couple of held-out tickets
print("\n" + "=" * 70)
print("QUALITATIVE EXAMPLES (fine-tuned outputs)")
print("=" * 70)
for i, (q, ref, pred) in enumerate(mistral_gen_scores["samples"][:2]):
    print(f"\n[{i+1}] Customer: {q}")
    print(f"    Reference reply : {ref[:160]}...")
    print(f"    Mistral (QLoRA) : {pred[:160]}...")

## 7. LLM classification vs BERT

The roadmap asks for "LLM classification." Since fine-tuned DistilBERT already hits ~0.999, the interesting question is: **how close does an off-the-shelf instruction LLM get with no training?** We run **zero-shot** and **few-shot** classification with Mistral-7B-Instruct and compare macro-F1 to the BERT baseline. (Expectation: respectable but below BERT, far slower.)

In [ ]:
# =============================================================
# 7. LLM classification (zero-shot + few-shot) vs BERT
# =============================================================
# No fine-tuning here: we use Mistral-7B-Instruct *as-is* and ask it
# to pick a category. This is the honest "LLM classifier" comparison
# against the fine-tuned DistilBERT (0.999 macro-F1) from Phase 3.
import re, gc, torch
from sklearn.metrics import accuracy_score, f1_score

# Reload Mistral-Instruct (base, 4-bit) for inference-only classification
cls_model, cls_tok = load_causal_lm(MISTRAL_MODEL, quantized=True)
cls_model.eval()

CAT_LIST_STR = ", ".join(CATEGORIES)

def _build_cls_prompt(query, shots=None):
    sys = (f"You are a support-ticket classifier. Classify the customer message into "
           f"exactly ONE of these categories: {CAT_LIST_STR}. "
           f"Reply with only the category name, nothing else.")
    msgs = [{"role": "user", "content": sys + "\n\nMessage: " + query + "\nCategory:"}]
    if shots:
        demo = "".join(f"Message: {q}\nCategory: {c}\n\n" for q, c in shots)
        msgs = [{"role": "user", "content": sys + "\n\n" + demo + "Message: " + query + "\nCategory:"}]
    return build_text(cls_tok, msgs[0]["content"])  # reuse the generation formatter (prompt only)

def _parse_category(text):
    up = text.upper()
    for c in CATEGORIES:                 # exact token match first
        if re.search(rf"\b{re.escape(c)}\b", up):
            return c
    return "UNKNOWN"

@torch.no_grad()
def classify_batch(df, shots=None):
    preds = []
    for q in df["instruction"].tolist():
        prompt = _build_cls_prompt(q, shots)
        out = generate(cls_model, cls_tok, prompt, max_new_tokens=16)
        preds.append(_parse_category(out))
    return preds

# Few-shot demonstrations: one example per category (from TRAIN, not eval)
few_shot = (train_df.groupby("category").first().reset_index()
            [["instruction", "category"]].values.tolist())
few_shot = [(q, c) for q, c in few_shot]

y_true = cls_eval_df["category"].tolist()

print(f"Zero-shot classifying {len(cls_eval_df)} tickets with Mistral-Instruct...")
zs_pred = classify_batch(cls_eval_df, shots=None)
print(f"Few-shot ({len(few_shot)} demos) classifying...")
fs_pred = classify_batch(cls_eval_df, shots=few_shot)

llm_cls_results = {
    "zero_shot_acc": accuracy_score(y_true, zs_pred),
    "zero_shot_f1":  f1_score(y_true, zs_pred, average="macro", labels=CATEGORIES, zero_division=0),
    "few_shot_acc":  accuracy_score(y_true, fs_pred),
    "few_shot_f1":   f1_score(y_true, fs_pred, average="macro", labels=CATEGORIES, zero_division=0),
}
print("\nLLM classification vs BERT baseline:")
print(f"  Zero-shot : acc={llm_cls_results['zero_shot_acc']:.4f}  macroF1={llm_cls_results['zero_shot_f1']:.4f}")
print(f"  Few-shot  : acc={llm_cls_results['few_shot_acc']:.4f}  macroF1={llm_cls_results['few_shot_f1']:.4f}")
print(f"  DistilBERT (Phase 3 fine-tuned): macroF1=0.9999  <-- still the winner, far cheaper")

# Free the classifier model
del cls_model
gc.collect(); torch.cuda.empty_cache()

## 8. LoRA vs QLoRA — the efficiency tradeoff

QLoRA = LoRA on top of a **4-bit quantized** base model. The claim is that 4-bit quantization is nearly free in quality but slashes VRAM — which is exactly what lets a 7B model fine-tune on a 16 GB T4. We measure it directly on **Phi-2** (small enough that fp16 LoRA also fits, so the comparison is apples-to-apples): peak VRAM, wall-clock training time, and ROUGE-L.

In [ ]:
# =============================================================
# 8. LoRA (fp16) vs QLoRA (4-bit) — measured on Phi-2
# =============================================================
# We already trained Phi-2 with QLoRA in section 4 and recorded its
# stats (phi2_qlora_stats). Here we train the SAME model + data with
# *plain LoRA in fp16* (no quantization) and compare VRAM / time /
# quality. We benchmark on Phi-2 (2.7B) because fp16 LoRA on a 7B
# barely fits a T4 — the whole point of QLoRA is that the 7B *needs*
# 4-bit. (Mistral is therefore QLoRA-only above.)

import gc, torch

lora_qlora_rows = []

# Row 1: the QLoRA run from section 4
lora_qlora_rows.append({
    "method": "QLoRA (4-bit)",
    "peak_vram_gb": phi2_qlora_stats["peak_vram_gb"],
    "train_time_s": phi2_qlora_stats["train_time_s"],
    "rougeL": phi2_gen_scores["rougeL_ft"],
})

# Row 2: train Phi-2 with fp16 LoRA (quantized=False)
print("Training Phi-2 with fp16 LoRA (no quantization)...")
lora_model, lora_tok = load_causal_lm(PHI2_MODEL, quantized=False)
lora_trainer, lora_time, lora_vram = run_sft(
    lora_model, lora_tok, gen_train_df, run_name="phi2_lora_fp16")
lora_rougeL = evaluate_rouge(lora_trainer.model, lora_tok, gen_eval_df,
                             n=min(80, N_EVAL))["rougeL"]
lora_qlora_rows.append({
    "method": "LoRA (fp16)",
    "peak_vram_gb": lora_vram,
    "train_time_s": lora_time,
    "rougeL": lora_rougeL,
})

# Free the fp16 model
del lora_model, lora_trainer
gc.collect(); torch.cuda.empty_cache()

cmp_df = pd.DataFrame(lora_qlora_rows)
print("\nLoRA vs QLoRA (Phi-2):")
print(cmp_df.to_string(index=False))
vram_ratio = lora_qlora_rows[0]["peak_vram_gb"] / max(lora_qlora_rows[1]["peak_vram_gb"], 1e-6)
print(f"\nQLoRA used {vram_ratio:.0%} of the fp16-LoRA peak VRAM; "
      f"ROUGE-L delta = {lora_qlora_rows[0]['rougeL'] - lora_qlora_rows[1]['rougeL']:+.4f}")

## 9. Results table → `experiments/phase4_results.csv`

Collate everything (generation ROUGE, classification F1, LoRA/QLoRA efficiency) into the project's experiment-tracker format, alongside the Phase 1 / Phase 3 baselines for context.

In [ ]:
# =============================================================
# 9. Collate all Phase-4 results -> experiments/phase4_results.csv
# =============================================================
import pandas as pd, os

rows = []

# --- Generation (ROUGE-L: base vs fine-tuned) ---
for tag, sc in [("Phi-2 (2.7B)", phi2_gen_scores), ("Mistral-7B-Instruct", mistral_gen_scores)]:
    rows.append({"task": "generation", "model": tag, "variant": "base (zero-shot)",
                 "metric": "ROUGE-L", "score": round(sc.get("rougeL_base", float('nan')), 4)})
    rows.append({"task": "generation", "model": tag, "variant": "QLoRA fine-tuned",
                 "metric": "ROUGE-L", "score": round(sc.get("rougeL_ft", float('nan')), 4)})

# --- Classification (macro-F1: LLM vs BERT baseline) ---
rows.append({"task": "classification", "model": "Mistral-7B-Instruct", "variant": "zero-shot",
             "metric": "macro-F1", "score": round(llm_cls_results.get("zero_shot_f1", float('nan')), 4)})
rows.append({"task": "classification", "model": "Mistral-7B-Instruct", "variant": "few-shot",
             "metric": "macro-F1", "score": round(llm_cls_results.get("few_shot_f1", float('nan')), 4)})
rows.append({"task": "classification", "model": "DistilBERT", "variant": "fine-tuned (Phase 3)",
             "metric": "macro-F1", "score": 0.9999})
rows.append({"task": "classification", "model": "TF-IDF + LogReg", "variant": "baseline (Phase 1)",
             "metric": "macro-F1", "score": 0.9955})

# --- Efficiency (LoRA vs QLoRA, measured on Phi-2) ---
for r in lora_qlora_rows:
    rows.append({"task": "efficiency", "model": "Phi-2 (2.7B)", "variant": r["method"],
                 "metric": "peak_VRAM_GB", "score": round(r["peak_vram_gb"], 2)})
    rows.append({"task": "efficiency", "model": "Phi-2 (2.7B)", "variant": r["method"],
                 "metric": "train_time_s", "score": round(r["train_time_s"], 1)})
    rows.append({"task": "efficiency", "model": "Phi-2 (2.7B)", "variant": r["method"],
                 "metric": "ROUGE-L", "score": round(r["rougeL"], 4)})

results_phase4 = pd.DataFrame(rows)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
out_csv = os.path.join(EXPERIMENTS_DIR, "phase4_results.csv")
results_phase4.to_csv(out_csv, index=False)
print(f"✅ Wrote {len(results_phase4)} rows -> {out_csv}\n")

# Pretty pivot for the writeup
with pd.option_context("display.max_rows", None, "display.width", 120):
    print(results_phase4.to_string(index=False))

## 10. Save adapters + model cards

Persist the trained **LoRA adapters** (small — a few MB each) and write a **model card** for each, matching the PM-deliverable convention. Adapter weights are gitignored; the `README.md` model cards are committed.

In [ ]:
# =============================================================
# 10. Write model cards for every saved adapter
# =============================================================
# Adapters themselves were already saved inline by finalize_run()
# right after each model trained (so we never hold two big models in
# VRAM at once). Here we just emit a human-readable model card per
# run from ADAPTER_REGISTRY. The card .md files are committed; the
# adapter weights are gitignored.
import os, json

def write_model_card(name, info):
    m = info["metrics"]
    card = f"""# Model card — {name}

- **Base model**: `{info['base_model']}`
- **Task**: {info['task']}
- **Method**: {m.get('method', 'QLoRA (4-bit) + LoRA adapter')}
- **Dataset**: Bitext Customer Support ({N_TRAIN_GEN} train / {N_EVAL} eval subset)
- **LoRA**: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, target=all-linear
- **Train config**: epochs={EPOCHS}, lr={LEARNING_RATE}, max_seq_len={MAX_SEQ_LEN}

## Metrics
```json
{json.dumps(m, indent=2)}
```

## How to load
```python
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
model = AutoPeftModelForCausalLM.from_pretrained("{info['dir']}", device_map="auto")
tok = AutoTokenizer.from_pretrained("{info['dir']}")
```

## Intended use
Drafting customer-support replies for agent review (assist, not full automation).
Not validated for autonomous deployment — see ../METRICS.md for the human-eval rubric.
"""
    path = os.path.join(info["dir"], "README.md")
    with open(path, "w") as f:
        f.write(card)
    print(f"✅ model card -> {path}")

if not ADAPTER_REGISTRY:
    print("No adapters in ADAPTER_REGISTRY — did the training sections run?")
for nm, info in ADAPTER_REGISTRY.items():
    write_model_card(nm, info)

## 11. Conclusions — the PM verdict

_Fill the bracketed numbers in from your run; the structure below is the argument to make._

**Generation quality.** Fine-tuning lifted ROUGE-L from `[base]` → `[finetuned]` for Phi-2 and `[base]` → `[finetuned]` for Mistral-7B. Sample outputs show the fine-tuned models adopt the dataset's house style (placeholder tokens like `{{Order Number}}`, polite closing) that the base models miss.

**Classification.** Zero-shot Mistral scored `[zs]` macro-F1, few-shot `[fs]` — versus **0.999 for fine-tuned DistilBERT** at a fraction of the cost. The LLM is *not* the right tool for this classification task: it is slower, more expensive, and no more accurate. This reinforces the Phase 1–3 finding.

**LoRA vs QLoRA.** On Phi-2, QLoRA used `[x]%` of the VRAM of fp16 LoRA for a `[y]` ROUGE difference — i.e. 4-bit quantization is nearly free in quality and is what makes 7B fine-tuning fit on a T4 at all.

**Where Phase 4 changes the recommendation:**
- For **routing/classification** → still ship TF-IDF + Logistic Regression (Phase 1). Nothing here beats it on cost-adjusted accuracy.
- For **drafting agent replies** → a QLoRA-fine-tuned 7B is the first approach that produces usable, on-brand responses. This is the new capability that justifies an LLM in the stack — as an **assist** (draft for a human agent), not full automation, until generation quality is human-evaluated at scale.

**Next (Phase 5):** RAG over historical responses to ground generation in real resolutions, and a human-eval rubric on a 100-response sample (see `METRICS.md`).